# Lab 02: Muscle Mechanics — The Actuators

**Computational Sensorimotor Control** | Week 2 | Module 1: The Biological Plant

In this lab you will build the `Muscle` class that implements the Gribble et al. (1998) muscle model. By the end, you will have six muscle objects that can be driven with direct activation inputs to produce joint torques.

**Pipeline reminder:**  Activation → Force-Length → Calcium Filtering → Force-Velocity → + Passive Stiffness → Total Force → Joint Torque

In Weeks 2–3, activation a(t) is set directly. In Week 4, it will emerge from the λ threshold law.

---
## Part 0: Setup

In [ ]:
# === Setup ===
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Model parameters (Gribble et al. 1998)
# Force-length
c = 0.112  # mm^-1, recruitment gradient (Feldman & Orlovsky 1972)
G = 20.0   # mm, neural drive gain (converts a∈[0,1] to effective displacement)

# Calcium kinetics
TAU = 0.015  # 15 ms time constant (Hainaut et al. 1981)

# Force-velocity (Joyce & Rack 1969)
f1, f2, f3, f4 = 0.82, 0.50, 0.43, 58.0  # s/m

# PCSA values → ρ (N, since scaling = 1 N/cm²) (Winters & Woo 1990)
MUSCLE_PARAMS = {
    'pectoralis':     {'rho': 14.9, 'k': 258.5, 'type': 'shoulder_flexor'},
    'deltoid':        {'rho': 14.9, 'k': 258.5, 'type': 'shoulder_extensor'},
    'biceps_long':    {'rho': 11.0, 'k': 190.9, 'type': 'elbow_flexor'},
    'triceps_lat':    {'rho': 12.1, 'k': 209.9, 'type': 'elbow_extensor'},
    'biceps_short':   {'rho':  2.1, 'k':  36.5, 'type': 'biarticular_flexor'},
    'triceps_long':   {'rho':  6.7, 'k': 116.3, 'type': 'biarticular_extensor'},
}

# Moment arms (m)
# Extensors: constant. Flexors: angle-dependent (we use midrange value for now)
MOMENT_ARMS = {
    'shoulder_flexor':         {'shoulder':  0.04, 'elbow': 0.0},
    'shoulder_extensor':       {'shoulder': -0.04, 'elbow': 0.0},
    'elbow_flexor':            {'shoulder':  0.0,  'elbow':  0.02},
    'elbow_extensor':          {'shoulder':  0.0,  'elbow': -0.02},
    'biarticular_flexor':      {'shoulder':  0.028,'elbow':  0.025},
    'biarticular_extensor':    {'shoulder': -0.035,'elbow': -0.02},
}

print('Setup complete. 6 muscles defined.')

---
## Part 1: The Exponential Force-Length Relationship (Equation 4.1)

$$\tilde{M} = \rho \left[ e^{cA} - 1 \right]$$

where $\rho$ is the PCSA-based magnitude, $c = 0.112$ mm$^{-1}$ is the recruitment gradient, and $A = G \times a$ is the effective activation. The neural drive gain $G = 20$ mm converts the dimensionless activation $a \in [0,1]$ to an effective displacement in millimeters (see lecture notes, Section 4).

### Task 1.1
COMPLETE the function `force_length` below.

In [ ]:
def force_length(a, rho, G=20.0, c=0.112):
    """Exponential force-length relationship.
    
    Parameters
    ----------
    A : float or array — activation level (≥ 0)
    rho : float — PCSA-based magnitude parameter
    c : float — recruitment gradient (mm^-1)
    
    Returns
    -------
    M_tilde : float or array — instantaneous muscle force (before calcium filtering)
    """
    # TODO: COMPLETE — implement ρ[exp(cA) - 1]
    M_tilde = ...  # your code here
    return M_tilde

### Task 1.2 — Verify and Plot
COMPLETE the cell below to plot force vs. activation for three different ρ values.

In [ ]:
# === Test ===
assert np.isclose(force_length(0, 10.0), 0.0), "Zero activation should give zero force"
assert force_length(1.0, 10.0) > 0, "Positive activation should give positive force"
print("Basic tests passed.")

# TODO: COMPLETE — plot M_tilde vs A for A in [0, 3] for rho = 2.1, 11.0, 14.9
A_range = np.linspace(0, 3, 200)
fig, ax = plt.subplots()
# your code here — three curves, one per rho, with labels

ax.set_xlabel('Activation A')
ax.set_ylabel('Force $\\tilde{M}$ (N)')
ax.set_title('Force-Length: Effect of ρ (PCSA)')
ax.legend()
plt.show()

### Question 1.1
Why does a muscle with larger ρ (larger PCSA) produce more force at the same activation level? What does the exponential shape imply about how force scales with activation — linearly or nonlinearly?

*Your answer here:*


---
## Part 2: Calcium Kinetics — Graded Force Development (Equation 5.1)

$$\tau^2 \ddot{M} + 2\tau \dot{M} + M = \tilde{M}$$

This critically damped second-order filter prevents force from changing instantaneously. We convert it to two first-order ODEs:

$$\dot{M}_1 = M_2$$
$$\dot{M}_2 = (\tilde{M} - M_1 - 2\tau M_2) / \tau^2$$

where $M_1 = M$ (filtered force) and $M_2 = \dot{M}$ (rate of change).

### Task 2.1
COMPLETE the function `calcium_dynamics` that computes the derivatives of the calcium filter state.

In [ ]:
def calcium_dynamics(state, M_tilde, tau=TAU):
    """Compute derivatives of the calcium kinetics filter.
    
    Parameters
    ----------
    state : array [M, M_dot] — current filter state
    M_tilde : float — instantaneous force input (from force_length)
    tau : float — time constant (s)
    
    Returns
    -------
    dstate : array [dM, dM_dot] — time derivatives
    """
    M, M_dot = state
    # TODO: COMPLETE — implement the two first-order ODEs
    dM = ...      # your code here
    dM_dot = ...  # your code here
    return np.array([dM, dM_dot])

### Task 2.2 — Simulate a Step Response
COMPLETE the simulation loop below. Drive the filter with a step input (M̃ jumps from 0 to 10 N at t = 20 ms) and plot the filtered output M(t).

Use simple Euler integration: `state += dt * calcium_dynamics(state, M_tilde)`

In [ ]:
dt = 0.0001  # 0.1 ms
T = 0.25     # 250 ms
t = np.arange(0, T, dt)

# Step input at 20 ms
M_tilde_input = np.where(t >= 0.02, 10.0, 0.0)

# TODO: COMPLETE — simulate the filter
state = np.array([0.0, 0.0])  # [M, M_dot]
M_output = np.zeros_like(t)

for i in range(len(t)):
    M_output[i] = state[0]
    # your code here — update state using Euler integration
    pass

# Plot
fig, ax = plt.subplots()
ax.plot(t * 1000, M_tilde_input, '--', color='gray', lw=2, label='Input $\\tilde{M}$ (step)')
ax.plot(t * 1000, M_output, '-', color='#E74C3C', lw=2.5, label='Output $M$ (filtered)')
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Force (N)')
ax.set_title(f'Calcium Kinetics: Step Response (τ = {TAU*1000:.0f} ms)')
ax.legend()
plt.show()

# Report 90% rise time
idx_90 = np.argmax(M_output >= 0.9 * M_output[-1])
print(f"~90% rise time: {t[idx_90]*1000 - 20:.0f} ms after step onset")

### Question 2.1
The filter reaches ~90% of its final value in approximately 60 ms after the step onset. Why does this delay matter for motor control? Think about what happens when the brain wants to *stop* a fast movement.

*Your answer here:*


---
## Part 3: The Force-Velocity Relationship (Equation 6.1)

$$F_{total} = M \cdot \left[ f_1 + f_2 \cdot \arctan(f_3 + f_4 \cdot \dot{l}) \right] + k(l - r)$$

The term in brackets is the force-velocity multiplier. We implement it separately first.

### Task 3.1
COMPLETE the function `force_velocity_multiplier`.

In [ ]:
def force_velocity_multiplier(dl_dt):
    """Sigmoidal force-velocity multiplier.
    
    Parameters
    ----------
    dl_dt : float or array — muscle contraction velocity (m/s)
        Positive = lengthening (eccentric), Negative = shortening (concentric)
    
    Returns
    -------
    Fv : float or array — force multiplier (dimensionless)
    """
    # TODO: COMPLETE — implement f1 + f2 * arctan(f3 + f4 * dl_dt)
    Fv = ...  # your code here
    return Fv

### Task 3.2 — Plot the Force-Velocity Curve
COMPLETE the plot of the multiplier vs. muscle velocity.

In [ ]:
# Test: at zero velocity, multiplier should be close to 1
Fv_zero = force_velocity_multiplier(0.0)
print(f"Multiplier at zero velocity: {Fv_zero:.3f}")

# TODO: COMPLETE — plot Fv vs dl_dt for dl_dt in [-0.1, 0.1] m/s
v_range = np.linspace(-0.1, 0.1, 500)
Fv = force_velocity_multiplier(v_range)

fig, ax = plt.subplots()
# your code here — plot, label lengthening vs shortening regions

ax.set_xlabel('Muscle velocity (m/s)')
ax.set_ylabel('Force multiplier')
ax.set_title('Force-Velocity Relationship')
plt.show()

### Question 3.1
(a) When a muscle is being stretched (lengthening/eccentric), is the force multiplier above or below 1? 

(b) Why is this force enhancement during lengthening useful for the control of movement? Think about what happens when the arm is perturbed.

*Your answer here:*


---
## Part 4: The Complete Muscle Class

Now assemble all components into a single `Muscle` class. Each muscle needs:
- Parameters: ρ, k (passive stiffness), moment arms, resting length
- State: [M, M_dot] (calcium filter state)
- Methods: `compute_force(activation, length, velocity)` and `reset()`

### Task 4.1
COMPLETE the Muscle class below.

In [ ]:
class Muscle:
    """Single muscle following the Gribble et al. (1998) model."""
    
    def __init__(self, name, rho, k, moment_arm_shoulder, moment_arm_elbow, rest_length):
        self.name = name
        self.rho = rho
        self.k = k                          # passive stiffness (N/m)
        self.moment_arm_shoulder = moment_arm_shoulder  # (m), signed
        self.moment_arm_elbow = moment_arm_elbow        # (m), signed
        self.rest_length = rest_length       # resting length (m)
        
        # Calcium filter state: [M, M_dot]
        self.state = np.array([0.0, 0.0])
    
    def reset(self):
        """Reset calcium filter state to zero."""
        self.state = np.array([0.0, 0.0])
    
    def compute_force(self, activation, length, velocity, dt):
        """Compute total muscle force for one timestep.
        
        Parameters
        ----------
        activation : float — activation level a(t) in [0, 1]
        length : float — current muscle length (m)
        velocity : float — current muscle velocity dl/dt (m/s)
        dt : float — timestep (s)
        
        Returns
        -------
        F_total : float — total muscle force (N)
        """
        # Step 1: Force-length (instantaneous)
        # TODO: COMPLETE — compute M_tilde using force_length(activation, self.rho, G)
        M_tilde = ...  # your code here
        
        # Step 2: Calcium filtering (update state with Euler integration)
        # TODO: COMPLETE — update self.state using calcium_dynamics
        dstate = ...   # your code here
        self.state += dt * dstate
        M = self.state[0]  # filtered force
        
        # Step 3: Force-velocity multiplier
        # TODO: COMPLETE — compute Fv
        Fv = ...  # your code here
        
        # Step 4: Passive stiffness
        # TODO: COMPLETE — compute F_passive = k * (length - rest_length)
        F_passive = ...  # your code here
        
        # Step 5: Total force
        # TODO: COMPLETE — F_total = M * Fv + F_passive
        F_total = ...  # your code here
        
        return F_total
    
    def joint_torques(self, F):
        """Convert muscle force to joint torques.
        
        Returns
        -------
        tau_shoulder, tau_elbow : float — torque contributions (Nm)
        """
        return self.moment_arm_shoulder * F, self.moment_arm_elbow * F

---
## Part 5: Instantiate All 6 Muscles

### Task 5.1
COMPLETE the cell below to create all 6 muscle objects. For resting lengths, use the muscle lengths at θ₁ = 45°, θ₂ = 90° (a comfortable mid-range posture). For this lab, we approximate resting length as a simple function of the moment arms and a reference configuration.

In [ ]:
# Approximate resting lengths at theta1=45°, theta2=90°
# For simplicity: rest_length = 0.26 m for shoulder muscles, 0.35 m for elbow muscles,
# 0.55 m for biarticular muscles (these are approximate MTC lengths at the reference posture)
REST_LENGTHS = {
    'shoulder_flexor': 0.26,
    'shoulder_extensor': 0.26,
    'elbow_flexor': 0.35,
    'elbow_extensor': 0.35,
    'biarticular_flexor': 0.55,
    'biarticular_extensor': 0.55,
}

# TODO: COMPLETE — create all 6 muscles
muscles = {}
for name, params in MUSCLE_PARAMS.items():
    mtype = params['type']
    ma = MOMENT_ARMS[mtype]
    rl = REST_LENGTHS[mtype]
    # your code here — create Muscle object and add to muscles dict
    pass

print(f"Created {len(muscles)} muscles:")
for name, m in muscles.items():
    print(f"  {name}: ρ={m.rho:.1f}, k={m.k:.1f}, r_sh={m.moment_arm_shoulder:.3f}, r_el={m.moment_arm_elbow:.3f}")

---
## Part 6: Drive the Muscles

### Task 6.1 — Single Muscle Step Response
COMPLETE the cell below to drive the **pectoralis** (shoulder flexor) with a step activation of a(t) = 0.5 starting at t = 50 ms. Plot force vs. time. Assume the muscle is at resting length and zero velocity (isometric contraction).

In [ ]:
dt = 0.0001
T = 0.4
t = np.arange(0, T, dt)

# Reset muscle state
pec = muscles['pectoralis']
pec.reset()

# Activation: step to 0.5 at 50 ms
activation = np.where(t >= 0.05, 0.5, 0.0)

# TODO: COMPLETE — simulate and record force
force_history = np.zeros_like(t)
for i in range(len(t)):
    F = pec.compute_force(activation[i], pec.rest_length, 0.0, dt)
    force_history[i] = F

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
ax1.plot(t * 1000, activation, 'k--', lw=1.5, label='Activation a(t)')
ax1.set_ylabel('Activation'); ax1.legend(); ax1.set_title('Pectoralis: Isometric Step Response')

ax2.plot(t * 1000, force_history, '#E74C3C', lw=2.5, label='Total force F(t)')
ax2.set_xlabel('Time (ms)'); ax2.set_ylabel('Force (N)'); ax2.legend()
plt.tight_layout()
plt.show()

### Task 6.2 — Compare All 6 Muscles
COMPLETE the cell below to drive all 6 muscles with the same step activation (a = 0.5 at t = 50 ms) and overlay their force responses on a single plot. Which muscle produces the most force? Which the least?

In [ ]:
dt = 0.0001
T = 0.4
t = np.arange(0, T, dt)
activation = np.where(t >= 0.05, 0.5, 0.0)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#E74C3C', '#3498DB', '#E67E22', '#2ECC71', '#9B59B6', '#1ABC9C']

# TODO: COMPLETE — loop over all muscles, simulate, plot
for (name, m), color in zip(muscles.items(), colors):
    m.reset()
    forces = np.zeros_like(t)
    for i in range(len(t)):
        forces[i] = m.compute_force(activation[i], m.rest_length, 0.0, dt)
    ax.plot(t * 1000, forces, color=color, lw=2, label=name)

ax.set_xlabel('Time (ms)'); ax.set_ylabel('Force (N)')
ax.set_title('All 6 Muscles: Isometric Step Response (a = 0.5)')
ax.legend(fontsize=9)
plt.show()

### Question 6.1
(a) Which muscle produces the most force? Why (in terms of its ρ parameter)?

(b) Do all muscles reach steady state at the same time? Why or why not?

*Your answer here:*


---
## Part 7: Velocity-Dependent Force

### Task 7.1 — Shortening vs. Lengthening
COMPLETE the cell below to show how muscle force depends on contraction velocity. For the **biceps long** (elbow flexor), compute steady-state force at three velocities: shortening (dl/dt = +0.05 m/s), isometric (0), and lengthening (dl/dt = −0.05 m/s). Use a constant activation of 0.5.

In [ ]:
dt = 0.0001
T = 0.5
t = np.arange(0, T, dt)
activation = np.where(t >= 0.05, 0.5, 0.0)

velocities = [-0.05, 0.0, 0.05]
labels = ['Shortening (−0.05 m/s)', 'Isometric (0)', 'Lengthening (+0.05 m/s)']
colors_v = ['#3498DB', '#333', '#E74C3C']

bic = muscles['biceps_long']

fig, ax = plt.subplots(figsize=(10, 5))

# TODO: COMPLETE — simulate for each velocity, plot
for vel, label, color in zip(velocities, labels, colors_v):
    bic.reset()
    forces = np.zeros_like(t)
    for i in range(len(t)):
        forces[i] = bic.compute_force(activation[i], bic.rest_length, vel, dt)
    ax.plot(t * 1000, forces, color=color, lw=2.5, label=label)

ax.set_xlabel('Time (ms)'); ax.set_ylabel('Force (N)')
ax.set_title('Biceps Long: Effect of Contraction Velocity (a = 0.5)')
ax.legend()
plt.show()

### Question 7.1
The lengthening contraction produces more force than isometric, which produces more than shortening. This is intrinsic velocity-dependent damping. In one sentence, explain why this property is useful for stabilizing the arm against unexpected perturbations.

*Your answer here:*


---
## Part 8: From Muscle Force to Joint Torque

### Task 8.1 — Agonist-Antagonist Torques
COMPLETE the cell below to compute the net shoulder torque when the pectoralis (flexor) is activated at a = 0.5 and the deltoid (extensor) is activated at a = 0.3. Both are isometric at resting length. Report the individual and net torques.

In [ ]:
dt = 0.0001
T = 0.4
t = np.arange(0, T, dt)

pec = muscles['pectoralis']; pec.reset()
delt = muscles['deltoid']; delt.reset()

# Activations
a_pec = np.where(t >= 0.05, 0.5, 0.0)
a_delt = np.where(t >= 0.05, 0.3, 0.0)

# TODO: COMPLETE — simulate both muscles, compute individual and net shoulder torques
tau_pec = np.zeros_like(t)
tau_delt = np.zeros_like(t)

for i in range(len(t)):
    F_pec = pec.compute_force(a_pec[i], pec.rest_length, 0.0, dt)
    F_delt = delt.compute_force(a_delt[i], delt.rest_length, 0.0, dt)
    tau_pec[i] = pec.joint_torques(F_pec)[0]   # shoulder torque
    tau_delt[i] = delt.joint_torques(F_delt)[0]  # shoulder torque

net_torque = tau_pec + tau_delt

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t * 1000, tau_pec, '#E74C3C', lw=2, label='Pectoralis (flexor)')
ax.plot(t * 1000, tau_delt, '#3498DB', lw=2, label='Deltoid (extensor)')
ax.plot(t * 1000, net_torque, 'k', lw=2.5, label='Net shoulder torque')
ax.axhline(0, color='gray', ls=':', alpha=0.5)
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Torque (Nm)')
ax.set_title('Agonist-Antagonist: Co-contraction')
ax.legend()
plt.show()

print(f"Steady-state pectoralis torque: {tau_pec[-1]:.4f} Nm")
print(f"Steady-state deltoid torque:    {tau_delt[-1]:.4f} Nm")
print(f"Net shoulder torque:            {net_torque[-1]:.4f} Nm")

### Question 8.1
Even though both muscles are active, the net torque is not zero. Why not? If you wanted the net torque to be exactly zero (pure co-contraction with no net rotation), what would you need to do?

*Your answer here:*


---
## Part 9: Challenge Problems (Optional)

### Challenge 9.1: Activation Ramp
Drive all 6 muscles with a slow ramp activation (0 → 1 over 500 ms). Plot the force of each muscle vs. time. Does the order in which muscles reach a given force level match what you'd expect from their ρ values?

In [ ]:
# your code here


### Challenge 9.2: Co-Contraction Stiffness
With the shoulder at rest (zero net torque), compute the net restoring torque when the shoulder is displaced by small angles (±5°) for two conditions: (a) low co-contraction (both muscles at a = 0.1) and (b) high co-contraction (both at a = 0.5). Verify that higher co-contraction produces a stiffer joint.

In [ ]:
# your code here


---
## Summary

You built the `Muscle` class with: `force_length`, `calcium_dynamics`, `force_velocity_multiplier`, `compute_force`, `joint_torques`.

**Key results:**
- Force scales exponentially with activation (not linearly) — the recruitment gradient c controls this
- Calcium kinetics impose a ~60 ms delay from activation to 90% force — the brain must plan ahead
- Lengthening contractions produce more force than shortening — intrinsic velocity-dependent damping
- Co-contraction produces joint stiffness without net torque — a control strategy, not a waste of energy

**Exposed for future use:** muscle length `l` and velocity `dl/dt` — these will be read by the Proprioceptor class in Week 9.

**Next week:** We connect these muscles to the skeleton using Lagrangian dynamics → the DynamicsEngine class.